# Jour 3 — Au-delà de la classification d’images

> **Version corrigée** — les cellules de solution sont signalées.


## Objectifs

- distinguer classification, localisation, détection et segmentation
- comprendre comment la cible change selon la tâche
- représenter une classe, une boîte englobante et un masque
- relier sortie, loss et métrique


## Comment suivre cette journée

Le jour 2 a construit les fondations : images numériques, neurone, couches, loss, train, validation et test. Aujourd'hui, nous explorons ce que ces briques permettent de faire.

Le parcours commence par les tâches visuelles, puis reprend les CNN pour observer leurs réponses et piloter leur entraînement. Les autres notebooks sont des ouvertures à choisir selon le rythme du groupe : PyTorch, séquences, reconstruction ou localisation.

Les démonstrations se lisent étape par étape avec l'enseignant. Il n'est pas nécessaire de mémoriser tous les noms ni de réécrire seul tous les blocs. Avant chaque calcul, repérez ce qui entre, ce qui sort et ce que le résultat permet de vérifier.

Le LSTM donne aujourd'hui un premier aperçu des séquences. Au jour 4, nous reprendrons leur mécanisme avec un SimpleRNN, une démonstration complète et un exercice où conserver l'ordre change la réponse.


## Une image ne définit pas à elle seule la tâche

Le réseau peut recevoir exactement la même image tout en devant produire des réponses très différentes. La **question posée** détermine la cible, la dernière couche, la fonction de perte et les métriques.

![Panorama des principales tâches visuelles](../../ressources/illustrations/jour_03/00_panorama_taches_visuelles.png)

- La **classification** attribue une classe à l’image entière.
- La **localisation** prédit la classe et la position d’un objet unique.
- La **détection** peut produire plusieurs classes et plusieurs boîtes dans une même image.
- La **segmentation** attribue une classe à chaque pixel.

La localisation et la détection utilisent souvent des **boîtes englobantes**, ou *bounding boxes*. La segmentation utilise un **masque**, c’est-à-dire une grille alignée sur les pixels.


## La forme de la cible annonce la difficulté du problème

![Représentations numériques des différentes cibles](../../ressources/illustrations/jour_03/00_sorties_cibles.png)

En classification multiclasse, la cible peut être un entier comme `2`. Le réseau produit un score par classe et une cross-entropy compare ces scores à la bonne classe.

Pour localiser un seul objet, la cible peut être un vecteur de quatre coordonnées : `[x1, y1, x2, y2]`. Une loss de régression comme la MAE mesure alors l’écart entre coordonnées réelles et prédites.

En segmentation binaire, la cible est une matrice de zéros et de uns. Chaque position indique si le pixel appartient ou non à l’objet. La comparaison s’effectue pixel par pixel.

Le mot **sortie** désigne ce que le réseau produit. La **cible** est la bonne réponse connue pendant l’entraînement. La loss mesure leur différence dans un format adapté.


### Loss et métrique ne répondent pas à la même question

La **loss** doit fournir un signal exploitable pour ajuster les poids. Elle est calculée à chaque batch et doit varier progressivement lorsque la prédiction change.

Une **métrique** sert surtout à interpréter ou comparer. L’accuracy convient à une classe, l’IoU à un recouvrement et une MAE à une distance numérique. Certaines métriques ne sont pas utilisées directement pour apprendre parce qu’elles sont trop brutales ou difficiles à optimiser.

Une bonne formulation associe donc :

1. une cible sans ambiguïté ;
2. une sortie de forme compatible ;
3. une loss qui guide l’apprentissage ;
4. des métriques qui décrivent la qualité utile.

Changer seulement la dernière couche sans adapter cible, loss et métriques conduit à une expérience incohérente.


## Les entrées restent des tenseurs d’images

![Dimensions d’un tenseur d’image](../../ressources/illustrations/jour_02/00_tenseur_axes.png)

Une image conserve les mêmes axes de hauteur, largeur et canaux. Ce qui change ici est surtout la sortie attendue.

Une classification peut produire un vecteur de dix scores. Une localisation simple peut produire quatre nombres. Une segmentation produit une grille de valeurs, généralement de même résolution ou d’une résolution liée à l’image.

Cette correspondance entre entrée et sortie doit être décidée avant l’architecture. On ne peut pas entraîner correctement un modèle si la cible n’est pas représentée de manière cohérente.


## Coordonnées absolues et coordonnées normalisées

![Boîte englobante décrite par quatre coordonnées](../../ressources/illustrations/jour_03/04_boite_englobante.png)

Des coordonnées **absolues** sont exprimées en pixels. Sur une image de largeur 32, `x=16` se situe au milieu.

Des coordonnées **normalisées** divisent chaque position par la largeur ou la hauteur. Elles restent entre 0 et 1 et dépendent moins de la résolution. Le coin gauche-haut est décrit par `(x1, y1)` et le coin droit-bas par `(x2, y2)`.

Il faut conserver partout la même convention. Inverser largeur et hauteur, utiliser le centre au lieu des coins ou mélanger pixels et valeurs normalisées crée une cible incohérente.


## Construire un exemple entièrement connu

La cellule suivante crée une image synthétique de 32 × 32 pixels contenant un rectangle. Comme nous l’avons généré nous-mêmes, nous connaissons exactement sa classe, sa boîte et son masque.

`Rectangle` sert seulement à dessiner la boîte sur le graphique. La véritable cible du modèle reste le vecteur `target_bbox`.


`np.zeros((32, 32))` crée une grille de zéros : un fond sombre. `image[y1:y2, x1:x2] = 1` remplit une zone claire. Le premier axe décrit les lignes, le second les colonnes ; la borne finale d'une tranche est exclue.

`np.array` range les quatre coordonnées dans un tableau. Diviser par 32 les exprime entre 0 et 1. Le masque est une autre grille : 1 à l'intérieur de la forme et 0 à l'extérieur. `Rectangle` dessine seulement son contour ; `x2-x1` est sa largeur et `y2-y1` sa hauteur. Les coordonnées choisies donnent ici un rectangle de 16 × 17 pixels.


In [ ]:
# NumPy gère les tableaux et les calculs numériques.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Crée le fond sombre sous forme de grille numérique.
image = np.zeros((32, 32), dtype="float32")
# Déduit le coin opposé ; la largeur et la hauteur sont égales.
x1, y1, x2, y2 = 7, 9, 23, 26
# Remplit la zone sélectionnée ; la dernière ligne et colonne sont exclues.
image[y1:y2, x1:x2] = 1.0

target_class = 1
# Range les deux coins de la boîte, normalisés par la taille de l’image.
target_bbox = np.array([x1 / 32, y1 / 32, x2 / 32, y2 / 32], dtype="float32")
# Crée une grille cible distincte, alignée sur les pixels.
target_mask = np.zeros((32, 32), dtype="float32")
# Marque les pixels de l’objet par 1 et conserve le fond à 0.
target_mask[y1:y2, x1:x2] = 1.0

# Réserve les emplacements graphiques, sans modifier les données.
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=2))
axes[0].set_title("Image et boîte")


In [ ]:
axes[1].imshow(target_mask, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("Masque cible")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
# Affiche le résultat des calculs précédents.
plt.show()

print("Classe :", target_class)
print("Boîte normalisée :", target_bbox.round(3))
print("Forme du masque :", target_mask.shape)


## Lire l’exemple

Le carré occupe les lignes `y1:y2` et les colonnes `x1:x2`. En NumPy, l’axe vertical vient avant l’axe horizontal : on écrit donc `image[y, x]`.

La boîte normalisée contient quatre nombres, alors que le masque contient 1 024 valeurs. Ces deux cibles décrivent le même objet avec des niveaux de précision différents.

Un masque indique précisément quels pixels appartiennent à l’objet. Une boîte est plus compacte, mais contient aussi du fond autour des formes non rectangulaires.


## À vous de jouer — créer une nouvelle cible

Créez une seconde image contenant un carré entre les coordonnées `(4, 6)` et `(20, 24)`.

Construisez :

- `image_exercice` ;
- `bbox_exercice` normalisée ;
- `mask_exercice` ;
- un affichage de l’image avec sa boîte.


In [ ]:
# Correction
image_exercice = np.zeros((32, 32), dtype="float32")
x1_e, y1_e, x2_e, y2_e = 4, 6, 20, 24
image_exercice[y1_e:y2_e, x1_e:x2_e] = 1.0

bbox_exercice = np.array(
    [x1_e / 32, y1_e / 32, x2_e / 32, y2_e / 32],
    dtype="float32",
)
mask_exercice = np.zeros((32, 32), dtype="float32")
mask_exercice[y1_e:y2_e, x1_e:x2_e] = 1.0

# Réserve un emplacement graphique.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image_exercice, cmap="gray", vmin=0, vmax=1)
ax.add_patch(Rectangle(
    (x1_e, y1_e), x2_e-x1_e, y2_e-y1_e,
    fill=False, edgecolor="red", linewidth=2,
))
ax.axis("off")
ax.set_title(f"Boîte normalisée : {bbox_exercice.round(2)}")
# Affiche le résultat des calculs précédents.
plt.show()


## À retenir

Une tâche visuelle est définie par la réponse attendue, pas seulement par les images disponibles. Avant de choisir un réseau, il faut savoir ce que représente chaque nombre de la cible et comment la qualité de la sortie sera mesurée.
